## Climate stripe plots

Climate stripes (also known as "warming stripes") were popularised by
climate scientist Ed Hawkins.  Each stripe represents one year, coloured
by how much warmer or cooler it was relative to a baseline climatology.
The result is a striking, minimal visualisation of long-term temperature
change.

This notebook builds a climate stripe plot for ERA5 2-metre temperature
at London using `ekp.timeseries.stripes`.

In [1]:
import earthkit.data as ekd
import earthkit.plots as ekp
import earthkit.transforms as ekt
import xarray as xr

xr.set_options(keep_attrs=True)

### Fetching an 85-year ERA5 time series

We request annual mean 2-metre temperature at London from 1940 to the
present.  The long date range is what makes climate stripes meaningful —
the trend only becomes visible over decades.

In [2]:
dataset = "reanalysis-era5-single-levels-timeseries"
request = {
    "variable": ["2m_temperature"],
    "location": {"longitude": -1, "latitude": 51.5},
    "date": ["1940-01-01/2025-12-31"],
    "data_format": "netcdf",
}

era5_timeseries = ekd.from_source("cds", dataset, request).to_xarray()
era5_timeseries

2026-03-19 10:39:37,119 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).
2026-03-19 10:39:37,120 INFO Request ID is 9ef3eec2-e6ab-4e34-98c6-0defd91151bb
2026-03-19 10:39:37,193 INF

<xarray.Dataset> Size: 9MB
Dimensions:     (valid_time: 753888)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 6MB 1940-01-01 ... 2025-12-31T23:...
    latitude    float64 8B ...
    longitude   float64 8B ...
Data variables:
    t2m         (valid_time) float32 3MB dask.array<chunksize=(753888,), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2024-09-02T04:48 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

### Computing annual anomalies

Climate stripes show *anomalies* — departures from a reference period —
rather than absolute temperatures.  We use `earthkit.transforms` to
compute the climatological mean and then the anomaly relative to the
1981–2010 baseline.

In [3]:
# Compute climatological annual mean and anomaly
clim = ekt.climatology.mean(era5_timeseries)
anomaly = ekt.climatology.anomaly(era5_timeseries, climatology=clim)
anomaly

<xarray.Dataset> Size: 1kB
Dimensions:     (valid_time: 86)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 688B 1940-12-31 ... 2025-12-31
    latitude    float64 8B 51.5
    longitude   float64 8B -1.0
Data variables:
    t2m         (valid_time) float32 344B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2024-09-02T04:48 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

### Plotting the stripes

`ekp.timeseries.stripes` takes the annual anomaly xarray object and
draws one coloured stripe per year.  The `cmap` argument controls the
diverging colour map — `"RdBu_r"` gives the classic blue-to-red warming
palette.

`chart.xticks` adds year labels at a chosen frequency, and
`chart.attribution` adds a small credit line.

In [ ]:
(ekp.timeseries.stripes(anomaly, cmap="RdBu_r")
 .xticks(frequency="10Y")
 .title(
     "ERA5 {variable_name} anomaly\n"
     "Location: {latitude:%Lt}, {longitude:%Ln} • Baseline: 1981–2010"
 )
 .attribution("Credit: C3S/ECMWF", location="lower right")
 .show())

### What's next?

The next notebook shows how to recreate the Copernicus Climate Pulse
global average temperature chart — a more complex time series that
overlays many years of data on shared axes.